# 04 — Lot-grain canonical surface (M1 restructure sketch)

**Decision under test**: demote the aggregated `int_*_holdings` views and make the
**lot grain** the exposed canonical surface. The cross-sync defect signals
(`zero_flap`, `id_handoff`, `merged_lots`, `zero_gross_lot`) are computed in a thin
holding-grain model — keys + flags only, no `any_value()` columns — and joined back
onto the lot rows. Aggregation to holding grain moves to consumption (`fct_holdings`).

Why: the across-record defects are *defined* at holding grain, so that computation must
stay upstream of the canonical surface; but the wide aggregate was consumer shaping, and
shipping it as canon froze columns (`net_amount`, key components, cost basis) out of reach.
Lot grain keeps full fidelity; Kimball: atomic grain is canon, aggregates are derived.

This notebook prototypes the two new pieces against the built warehouse and proves
**parity**: the thin signals reproduce exactly the flags the holdings views emit today.

In [1]:
import duckdb

con = duckdb.connect('../warehouse.duckdb', read_only=True)

FAMILIES = {
    'bank_fixed_incomes': 'quantity',
    'credit_fixed_incomes': 'quantity',
    'funds': 'quota_quantity',
    'treasure_titles': 'quantity',
    'variable_incomes': 'quantity',
}

## 1. Thin signals at holding grain

Same grouping and lag/lead the holdings views use today, but the output is only
`(snapshot_id, account_id, holding_key, cross_sync_flags)`. This becomes the
internal `int_*_holding_signals` model (view; swappable to incremental at scale).

In [2]:
def signals_sql(family, qty):
    return f"""
    WITH admitted AS (
        SELECT * FROM int_{family}_positions WHERE admission = 'admit'
    ),
    holding AS (
        SELECT
            snapshot_id, account_id,
            coalesce(natural_key, investment_id) AS holding_key,
            any_value(snapshot_created_at)       AS snapshot_created_at,
            sum({qty})                           AS quantity,
            sum(gross_amount)                    AS gross_amount,
            count(*)                             AS n_lots,
            bool_or(coalesce(gross_amount,0)=0)  AS has_zero_lot,
            array_agg(investment_id ORDER BY investment_id) AS investment_ids
        FROM admitted GROUP BY 1,2,3
    ),
    timeline AS (
        SELECT *,
            lag(gross_amount)   OVER win AS prev_gross,
            lead(gross_amount)  OVER win AS next_gross,
            lag(quantity)       OVER win AS prev_qty,
            lead(quantity)      OVER win AS next_qty,
            lag(investment_ids) OVER win AS prev_ids
        FROM holding
        WINDOW win AS (PARTITION BY account_id, holding_key ORDER BY snapshot_created_at, snapshot_id)
    )
    SELECT snapshot_id, account_id, holding_key,
        list_filter([
            CASE WHEN n_lots > 1 THEN 'merged_lots' END,
            CASE WHEN n_lots > 1 AND has_zero_lot THEN 'zero_gross_lot' END,
            CASE WHEN gross_amount = 0 AND prev_gross > 0 AND next_gross > 0
                      AND quantity = prev_qty AND quantity = next_qty THEN 'zero_flap' END,
            CASE WHEN len(list_filter(investment_ids, id -> NOT list_contains(prev_ids, id))) > 0
                  AND len(list_filter(prev_ids, id -> NOT list_contains(investment_ids, id))) > 0
                 THEN 'id_handoff' END
        ], f -> f IS NOT NULL) AS cross_sync_flags
    FROM timeline
    """

con.sql(f"SELECT * FROM ({signals_sql('variable_incomes', 'quantity')}) WHERE len(cross_sync_flags) > 0 LIMIT 5").df()

,snapshot_id,account_id,holding_key,cross_sync_flags
0,affc8527-5701-528e-91d2-eac379fbd7ac,de18239c-1649-578b-8644-371dbb2ae159,PETR4,[merged_lots]
1,12280d3b-336e-5588-a0f3-0165d3a1a9d6,de18239c-1649-578b-8644-371dbb2ae159,PETR4,[merged_lots]
2,234f5d0f-763e-57f8-9062-003eb6e1c3cb,de18239c-1649-578b-8644-371dbb2ae159,PETR4,[merged_lots]
3,1e0466c8-84fc-53a8-b130-7e7291721d0e,de18239c-1649-578b-8644-371dbb2ae159,PETR4,[merged_lots]
4,1f0c4479-35ab-53d5-acba-907b16126aff,de18239c-1649-578b-8644-371dbb2ae159,PETR4,[merged_lots]


## 2. Join-back: the exposed lot-grain surface

Canonical positions = every classified lot, with the holding's cross-sync flags
appended to its own `data_quality_flags`. LEFT JOIN: rejected/quarantined lots were
not in the admitted timeline, so their flags coalesce to the lot-level ones.
A lot carrying `zero_flap` reads as *"this lot belongs to a holding that flapped"*.

In [3]:
def exposed_sql(family, qty):
    return f"""
    SELECT
        l.*,
        list_distinct(l.data_quality_flags || coalesce(s.cross_sync_flags, [])) AS all_flags
    FROM int_{family}_positions l
    LEFT JOIN ({signals_sql(family, qty)}) s
      ON  l.snapshot_id = s.snapshot_id
      AND l.account_id  = s.account_id
      AND coalesce(l.natural_key, l.investment_id) = s.holding_key
    """

con.sql(f"""
    SELECT snapshot_id, account_id, investment_id, natural_key, admission,
           data_quality_flags AS lot_flags, all_flags
    FROM ({exposed_sql('variable_incomes', 'quantity')})
    WHERE list_contains(all_flags, 'zero_flap')
    LIMIT 5
""").df()

,snapshot_id,account_id,investment_id,natural_key,admission,lot_flags,all_flags
0,2a0887a8-dec3-5e3f-8a43-5d3eab2f61aa,3d18e495-9924-53db-8257-65d99f81f3dd,f491871b-6a2c-5bcb-a0fe-8f09a26b4352,VALE3,admit,[],[zero_flap]
1,ea8eef9d-4d84-5176-9aaa-057bfb11a11e,57c8133e-fb4d-565e-9a8f-dbae92d5f7c7,8124311427926871,NaN,admit,"[missing_key, missing:ticker, missing:isin_code]","[zero_flap, missing:ticker, missing:isin_code,..."
2,a3cbb896-0846-5179-a356-1f1498bf6f8b,e94781d0-56c1-5b4d-9fd1-a4c0d8e01374,16b4b28b-657d-5f6e-b9ce-06f0ba60533b,BBDC4,admit,[],[zero_flap]
3,ca2a4225-b9ad-5c75-b53e-1891257d14eb,0301b555-ac5a-5bbc-be7a-2b4e321e0ab5,6515843439509224,NaN,admit,"[missing_key, missing:ticker, missing:isin_code]","[zero_flap, missing:ticker, missing:isin_code,..."
4,5e3c6390-3db9-5550-82bd-497fd3a004d2,4cf12fb3-f41c-5afb-98fb-9ad46914a1c2,9596c2ee-3b47-5dc9-a6fd-ff0c9688070b,WEGE3,admit,[],[zero_flap]


## 3. Parity and integrity checks

For every family:
1. signals grain is unique on `(snapshot_id, account_id, holding_key)`;
2. each cross-sync flag fires on exactly as many holdings as the current
   `int_*_holdings` view reports — the restructure changes *where* flags live,
   not *what* they say;
3. the join-back preserves the lot count (no fanout, no loss).

In [4]:
for fam, qty in FAMILIES.items():
    sig = signals_sql(fam, qty)

    dup = con.sql(f"""SELECT count(*) FROM (
        SELECT snapshot_id, account_id, holding_key FROM ({sig})
        GROUP BY ALL HAVING count(*) > 1)""").fetchone()[0]
    assert dup == 0, (fam, 'grain', dup)

    for flag in ['merged_lots', 'zero_gross_lot', 'zero_flap', 'id_handoff']:
        new = con.sql(f"SELECT count(*) FROM ({sig}) WHERE list_contains(cross_sync_flags, '{flag}')").fetchone()[0]
        old = con.sql(f"SELECT count(*) FROM int_{fam}_holdings WHERE list_contains(data_quality_flags, '{flag}')").fetchone()[0]
        assert new == old, (fam, flag, new, old)
        if new:
            print(f'{fam:22s} {flag:15s} {new:5d}  parity ok')

    n_lots = con.sql(f'SELECT count(*) FROM int_{fam}_positions').fetchone()[0]
    n_join = con.sql(f'SELECT count(*) FROM ({exposed_sql(fam, qty)})').fetchone()[0]
    assert n_lots == n_join, (fam, 'fanout', n_lots, n_join)

print('ALL PARITY CHECKS PASS')

bank_fixed_incomes     merged_lots        45  parity ok
bank_fixed_incomes     id_handoff        120  parity ok
credit_fixed_incomes   merged_lots         7  parity ok
credit_fixed_incomes   id_handoff         12  parity ok
funds                  merged_lots       464  parity ok
funds                  zero_gross_lot     79  parity ok
treasure_titles        merged_lots       730  parity ok
variable_incomes       merged_lots      1729  parity ok
variable_incomes       zero_gross_lot     22  parity ok
variable_incomes       zero_flap          32  parity ok
ALL PARITY CHECKS PASS


## 4. Conclusion → M1 implementation

The thin signals reproduce the holdings flags exactly, and the join-back is fanout-free.
The restructure is a pure relocation of information, no semantic drift. Implementation:

| Action | Where |
|---|---|
| Add `int_*_holding_signals` (view, keys + flags, grain-tested) | `intermediate/signals/` |
| Rework macro: `holding_timeline` / `holding_data_quality_flags` → single `holding_signals()` | `macros/openfinance.sql` |
| Expose lots ⋈ signals as the final per-family positions surface | positions models or a thin view on top |
| **Delete** the five `int_*_holdings` models + yml entries | `intermediate/holdings/` |
| Move holding-grain uniqueness test onto signals; document flag semantics ("lot belongs to a holding that…") | `_canonical.yml` |

Open question for implementation: whether the exposed surface is the `int_*_positions`
table itself (flags appended in-model, but then the incremental can't see `lead()` for
late-arriving syncs) or a view layered on it (flags always current, chosen here).